# Preprocess PubTator evidence sentences

Candidate files use columns: `pmid`, `virus` (`name|taxid`), `disease` (`name|MESH:D...`), and `sentence`.


In [ ]:
import pickle
from pathlib import Path

# Precomputed article-level entity index from PubTator annotations.
# It stores PMID-level virus/disease lookup objects; this notebook uses it
# to select articles that contain both viral species and disease annotations.
ARTICLE_ENTITY_INDEX = Path("data/article_entity_index.pkl")
BIOCXML_DIR = Path("data/pubtator_biocxml")

virus_pmids_midhigh, pmid2disease, _ = pickle.load(open(ARTICLE_ENTITY_INDEX, "rb"))
target_pmids = set(virus_pmids_midhigh) & set(pmid2disease)


In [ ]:
xml_files = sorted(BIOCXML_DIR.rglob("*.XML"))
print(f"xml files: {len(xml_files)}")
print(f"target pmids: {len(target_pmids)}")


In [ ]:
import re
from lxml import etree
from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib

SENTENCE_RE = re.compile(r"[^.;]+[.;]?")


def clean_text(text):
    return str(text).replace("\t", " ").replace("\n", " ").replace('"', "'").strip()


def annotation_rows(passage, text, passage_offset):
    rows = []
    for ann in passage.findall("annotation"):
        entity_type = ann.findtext("infon[@key='type']")
        entity_id = ann.findtext("infon[@key='identifier']")
        loc = ann.find("location")
        if loc is None:
            continue
        start = int(loc.attrib["offset"]) - passage_offset
        end = start + int(loc.attrib["length"])
        if start < 0 or end > len(text):
            continue
        rows.append((entity_type, entity_id, start, end))
    return rows


def sentence_entities(text, annotations, start, end):
    viruses = set()
    diseases = set()
    for entity_type, entity_id, entity_start, entity_end in annotations:
        if entity_start < start or entity_end > end:
            continue
        span = text[entity_start:entity_end].strip()
        if not span or not entity_id:
            continue
        if entity_type == "Species" and entity_id.isdigit():
            viruses.add(f"{span}|{entity_id}")
        elif entity_type == "Disease" and entity_id.startswith("MESH:"):
            diseases.add(f"{span}|{entity_id}")
    return viruses, diseases


def iter_candidate_rows(xml_file, passage_types):
    context = etree.iterparse(str(xml_file), events=("end",), tag="document", recover=True)
    for _, elem in context:
        pmid = elem.findtext("id")
        if pmid not in target_pmids:
            elem.clear()
            while elem.getprevious() is not None:
                del elem.getparent()[0]
            continue

        for passage in elem.findall("passage"):
            passage_type = passage.findtext("infon[@key='type']")
            if passage_type not in passage_types:
                continue
            text_node = passage.find("text")
            if text_node is None or not text_node.text:
                continue
            text = text_node.text
            passage_offset_text = passage.findtext("offset")
            passage_offset = int(passage_offset_text) if passage_offset_text and passage_offset_text.strip() else 0
            annotations = annotation_rows(passage, text, passage_offset)

            for match in SENTENCE_RE.finditer(text):
                sentence = match.group().strip()
                if not sentence:
                    continue
                viruses, diseases = sentence_entities(text, annotations, match.start(), match.end())
                if viruses and diseases:
                    yield pmid, ",".join(sorted(viruses)), ",".join(sorted(diseases)), clean_text(sentence)

        elem.clear()
        while elem.getprevious() is not None:
            del elem.getparent()[0]


def process_xml_chunk(file_list, part_id, passage_types, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"part_{part_id:02d}.tsv"
    written = 0
    with open(output_path, "w", encoding="utf-8") as out:
        for xml_file in file_list:
            try:
                for pmid, viruses, diseases, sentence in iter_candidate_rows(xml_file, passage_types):
                    out.write(f"{pmid}\t{viruses}\t{diseases}\t{sentence}\n")
                    written += 1
            except Exception as exc:
                print(f"ERROR {xml_file}: {exc}")
    return written


def write_candidate_parts(output_dir, passage_types, n_jobs=32):
    chunks = [xml_files[i::n_jobs] for i in range(n_jobs)]
    tasks = [(chunk, i) for i, chunk in enumerate(chunks) if chunk]
    with tqdm_joblib(tqdm(total=len(tasks), mininterval=0.5)):
        written = Parallel(n_jobs=n_jobs, backend="loky", batch_size=1)(
            delayed(process_xml_chunk)(chunk, part_id, passage_types, output_dir)
            for chunk, part_id in tasks
        )
    print("written_lines:", sum(written))


In [ ]:
ABSTRACT_OUTPUT_DIR = Path("results/abstract_candidate_parts")
write_candidate_parts(
    output_dir=ABSTRACT_OUTPUT_DIR,
    passage_types={"abstract"},
    n_jobs=32,
)


In [ ]:
FULLTEXT_OUTPUT_DIR = Path("results/fulltext_candidate_parts")
write_candidate_parts(
    output_dir=FULLTEXT_OUTPUT_DIR,
    passage_types={"paragraph"},
    n_jobs=32,
)
